# B2_typescript: executable lesson notebook

Read [the guide](README.md). Offline by default; mock output is a contract fixture.

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

In [ ]:
import shutil
import subprocess

ts_folder = ROOT / "modules/bonus/B2_typescript"
node = shutil.which("node")
assert node and (ts_folder / "node_modules/tsx").exists(), (
    "Install Node.js and npm ci --prefix modules/bonus/B2_typescript first"
)

## 01_first_agent

```typescript
/** First TypeScript Agent. Run: npx tsx examples/01_first_agent.ts --mock. Expected Welcome to Nimbus. Mock cost $0; no resources. */
import {run} from '../common.js';
console.log(await run(process.argv.includes('--mock'), false));

```

In [ ]:
result = subprocess.run(
    [node, "--import", "tsx", "examples/01_first_agent.ts", "--mock"],
    cwd=ts_folder,
    check=True,
    capture_output=True,
    text=True,
    timeout=45,
)
print(result.stdout)

## 02_typed_tool

```typescript
/** Validated tool. Run: npx tsx examples/02_typed_tool.ts --mock. Expected totalCents 35600; $0, no AWS. */
import {quoteTool} from '../common.js';
console.log(await quoteTool.invoke({travelers: 2}));

```

In [ ]:
result = subprocess.run(
    [node, "--import", "tsx", "examples/02_typed_tool.ts", "--mock"],
    cwd=ts_folder,
    check=True,
    capture_output=True,
    text=True,
    timeout=45,
)
print(result.stdout)

## 03_provider_and_result

```typescript
/** Provider boundary and usage. Run: npx tsx examples/03_provider_and_result.ts --mock. Mock $0; live model token charges. */
import {run} from '../common.js';
console.log(await run(process.argv.includes('--mock')));

```

In [ ]:
result = subprocess.run(
    [node, "--import", "tsx", "examples/03_provider_and_result.ts", "--mock"],
    cwd=ts_folder,
    check=True,
    capture_output=True,
    text=True,
    timeout=45,
)
print(result.stdout)

## typescript_booking

```typescript
/** Nimbus quote workflow. Run: npm run demo. Expected quote only; mock cost $0; creates no resources. */
import {run} from '../common.js';
try { console.log(await run(process.argv.includes('--mock'))); }
catch (error) { console.error('Check provider settings and scripts/check_env.py:', error instanceof Error ? error.message : 'Invocation failed'); process.exitCode = 1; }

```

In [ ]:
result = subprocess.run(
    [node, "--import", "tsx", "use_case/typescript_booking.ts", "--mock"],
    cwd=ts_folder,
    check=True,
    capture_output=True,
    text=True,
    timeout=45,
)
print(result.stdout)